In [2]:
print("Hello LLM provider!")

Hello LLM provider!


In [16]:
from dataclasses import dataclass
from openai import OpenAI
import os

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests accross all inference providers """

    name: str
    env_var: str
    is_true: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("Groq","GROQ_API_KEY",True,"https://api.groq.com/openai/v1","openai/gpt-oss-120b"),
    Provider("OpenRouter","OPENROUTER_API_KEY",True,"https://openrouter.ai/api/v1","stealth/space-bunny-alpha")
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")


def build_client(provider: Provider) -> OpenAI:

    api_key = os.getenv(provider.env_var)


    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.


In [17]:
def model_reply(prompt: str) -> str :

    provider = select_provider()
    print(f"Using {provider.name} provider and {provider.model} model")
    client = build_client(provider)

    reply = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return reply.choices[0].message.content

In [18]:
prompt = "Who was the prime minister of UK before 2020? Answer in one line"

try:
    print(model_reply(prompt))
except Exception as e:
    print(f"Error: {e}")

Using Groq provider and openai/gpt-oss-120b model
Boris Johnson was the Prime Minister of the United Kingdom in 2019, serving immediately before 2020.


In [19]:
def chat_reply(conversation: list[dict]) -> str:

    provider = select_provider()
    client = build_client(provider)

    response = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=conversation
    )

    return response.choices[0].message.content

In [20]:
conversation = []

In [21]:
conversation.append({"role":"user","content":"who was the prime minister of UK before 2020?"})

In [22]:
llm_reply = chat_reply(conversation)
print(llm_reply)

conversation.append({
    "role": "assistant",
    "content": llm_reply,
})

The Prime Minister of the United Kingdom **immediately before the year 2020 was Boris Johnson**. He took office on 24 July 2019 after winning the Conservative Party leadership contest following the resignation of Theresa May.

For


In [23]:
conversation

[{'role': 'user', 'content': 'who was the prime minister of UK before 2020?'},
 {'role': 'assistant',
  'content': 'The Prime Minister of the United Kingdom **immediately before the year\u202f2020 was Boris Johnson**. He took office on 24\u202fJuly\u202f2019 after winning the Conservative Party leadership contest following the resignation of Theresa\u202fMay.\n\nFor'}]

In [27]:
conversation.append({
    "role":"user",
    "content": "what was GDP at that time? answer in 2 line"
})

In [32]:
conversation

[{'role': 'user', 'content': 'who was the prime minister of UK before 2020?'},
 {'role': 'assistant',
  'content': 'The Prime Minister of the United Kingdom **immediately before the year\u202f2020 was Boris Johnson**. He took office on 24\u202fJuly\u202f2019 after winning the Conservative Party leadership contest following the resignation of Theresa\u202fMay.\n\nFor'},
 {'role': 'user', 'content': 'what was GDP at that time? answer in 2 line'}]

In [33]:
llm_reply_2 = chat_reply(conversation)

print(llm_reply_2)

- UK nominal GDP in 2019 (the year before 2020) was about **£2.2 trillion** (≈ US $2.8 trillion).  
- In 2020, amid the COVID‑19 recession, it fell to roughly **£1.9 trillion** (≈ US $2.5 trillion).
